In [ ]:
######################################################### Endothelial产图的code   ######### 
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
import pandas as pd
#sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_header()
#sc.settings.set_figure_params(dpi=80, facecolor="white")
import scvelo as scv
import cellrank as cr
from anndata import AnnData
import matplotlib.pyplot as plt
import dynamo as dyn
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors
import stream as st
import bbknn
import datetime
import harmonypy as hm


In [ ]:
adata = sc.read_h5ad('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_tissuebatch.h5ad')


In [ ]:
#adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_ds_800_ctgeneral.h5ad')
# adata.shape


In [ ]:
# adata = sc.read_h5ad('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/origin/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_ds_pan_each2k_origin.h5ad')
adata = sc.read_h5ad('./data/scalex/fibro/human/trajectory/data/FibAtlas_CelltypePan_BatT_45050.h5ad')   # 47646


In [ ]:
# adata = sc.read_h5ad('/./data/scalex/fibro/human/trajectory/data//adata_meta_fib.h5ad')

adata = sc.read_h5ad('./data/scalex/endo/human/trajectory/data/EndoAtlas_CelltypePan_BatT_19011.h5ad')


In [ ]:
adata

In [ ]:
len(adata.obs['leiden'].unique())


In [ ]:
os.getcwd()
os.chdir('external_inputs/wangyuting')


In [ ]:
# adata.obs['sample'].value_counts()


In [ ]:
adata.obs[['leiden','Celltype','Celltype_general','Celltype_sub']].drop_duplicates().sort_values('Celltype')


In [ ]:
###### 尝试下把那几个零散的 Leiden 去掉后再来用 meta cell 做轨迹分析, 

adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
adata.obs['Celltype_sub'] = adata.obs['Celltype_sub'].astype(str)
adata.obs['Celltype_general'] = adata.obs['Celltype_general'].astype(str)
adata.obs['label'] = adata.obs['Celltype_general']
leiden_list = list(adata.obs['leiden'].unique())
del_leiden = ['11', '19','22','25']
include_leiden = [item for item in leiden_list if item not in set(del_leiden)]
adata = adata[adata.obs['leiden'].isin(include_leiden)]


In [ ]:
adata.obs['leiden'].unique()


In [ ]:
adata

In [ ]:
# sc.pl.umap(adata, color=['tissue','Celltype','leiden','Celltype_general','Celltype_sub'], legend_loc='on data',ncols=3 )  # , legend_loc='on data'


In [ ]:
# adata.obs['Celltype_general'].value_counts()


In [ ]:
adata.obs['Celltype_general'].value_counts()


In [ ]:
adata.obs['tissue'].value_counts()


In [ ]:
tissue_list = list(adata.obs['tissue'].unique())
del_tissue = ['nose','spleen','glands','carcinoma']
include_tissue = [item for item in tissue_list if item not in set(del_tissue)]
adata = adata[adata.obs['tissue'].isin(include_tissue)]



In [ ]:
############### 产 meta-cell, 之前30个细胞一个meta-cell 有点太严格了，可以考虑 20个
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix, vstack
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def create_meta_cells(adata, n_cells_per_meta=30, min_cells_to_keep=15, 
                      leiden_col='leiden', sample_col='sample', 
                      use_rep='X_pca', random_state=42):
    """
    将单细胞聚合成元细胞
    
    参数:
    ----------
    adata : AnnData
        原始单细胞数据
    n_cells_per_meta : int, 默认=30
        每个元细胞包含的单细胞数量
    min_cells_to_keep : int, 默认=15
        当剩余细胞不足n_cells_per_meta但大于此值时，聚合成一个元细胞
    leiden_col : str, 默认='leiden'
        leiden聚类列名
    sample_col : str, 默认='sample'
        样本列名
    use_rep : str, 默认='X_pca'
        用于计算最近邻的嵌入表示
    random_state : int, 默认=42
        随机种子
    
    返回:
    ----------
    adata_meta : AnnData
        聚合后的元细胞数据
    meta_info : pd.DataFrame
        元细胞映射信息
    """
    
    # 确保有需要的列
    if leiden_col not in adata.obs.columns:
        raise ValueError(f"列 '{leiden_col}' 不在 adata.obs 中")
    if sample_col not in adata.obs.columns:
        raise ValueError(f"列 '{sample_col}' 不在 adata.obs 中")
    
    # 确保有降维数据
    if use_rep not in adata.obsm_keys():
        print(f"警告: '{use_rep}' 不在 adata.obsm 中，使用 PCA 降维...")
        sc.tl.pca(adata, random_state=random_state)
        use_rep = 'X_pca'
    
    np.random.seed(random_state)
    
    # 存储结果的列表
    meta_X_list = []
    meta_obs_list = []
    meta_var = adata.var.copy()
    meta_cell_mapping = []  # 记录元细胞到原始细胞的映射
    
    # 创建进度条
    print("开始聚合元细胞...")
    print(f"每个元细胞包含 {n_cells_per_meta} 个单细胞")
    print(f"最小保留细胞数: {min_cells_to_keep}")
    
    # 获取所有唯一的 leiden-sample 组合
    adata.obs['leiden_sample'] = adata.obs[leiden_col].astype(str) + '_' + adata.obs[sample_col].astype(str)
    unique_groups = adata.obs['leiden_sample'].unique()
    
    total_meta_cells = 0
    discarded_cells = 0
    
    # 为每个 leiden-sample 组合处理
    for group in tqdm(unique_groups, desc="处理分组"):
        # 获取该分组的细胞索引
        group_mask = adata.obs['leiden_sample'] == group
        group_indices = np.where(group_mask)[0]
        group_cells_count = len(group_indices)
        
        if group_cells_count == 0:
            continue
        
        # 获取该分组的原始细胞数据
        group_adata = adata[group_indices]
        group_X = group_adata.X
        group_obs = group_adata.obs.copy()
        group_embeddings = group_adata.obsm[use_rep]
        
        # 计算最近邻
        if group_cells_count > n_cells_per_meta:
            # 使用k+1因为包含自身
            n_neighbors = min(n_cells_per_meta, group_cells_count)
            nn_model = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
            nn_model.fit(group_embeddings)
            
            # 初始化细胞选择标记
            selected_mask = np.zeros(group_cells_count, dtype=bool)
            current_indices = list(range(group_cells_count))
            
            # 随机打乱顺序以避免偏倚
            np.random.shuffle(current_indices)
            
            # 创建元细胞
            i = 0
            while i < len(current_indices) and np.sum(~selected_mask) >= n_cells_per_meta:
                if selected_mask[current_indices[i]]:
                    i += 1
                    continue
                
                # 获取当前细胞及其最近邻
                current_idx = current_indices[i]
                distances, neighbor_indices = nn_model.kneighbors(
                    group_embeddings[current_idx:current_idx+1]
                )
                
                # 获取尚未被选择的最近邻
                available_neighbors = []
                for idx in neighbor_indices[0]:
                    if not selected_mask[idx]:
                        available_neighbors.append(idx)
                    if len(available_neighbors) >= n_cells_per_meta:
                        break
                
                # 如果找到了足够的细胞
                if len(available_neighbors) == n_cells_per_meta:
                    # 标记这些细胞已被选择
                    for idx in available_neighbors:
                        selected_mask[idx] = True
                    
                    # 聚合这些细胞
                    meta_cell_indices = available_neighbors
                    
                    # 计算元细胞的表达（平均值）
                    if isinstance(group_X, csr_matrix):
                        meta_X = group_X[meta_cell_indices].mean(axis=0)
                    else:
                        meta_X = np.mean(group_X[meta_cell_indices], axis=0)
                    
                    meta_X_list.append(meta_X)
                    
                    # 创建元细胞的观测数据（取众数或平均值） 
                    meta_obs = {}
                    for col in group_obs.columns:
                        if col == 'leiden_sample':
                            continue
                        if group_obs[col].dtype == 'object' or group_obs[col].dtype.name == 'category':
                            # 对于分类变量，取众数
                            values = group_obs.iloc[meta_cell_indices][col]
                            meta_obs[col] = values.mode()[0] if not values.mode().empty else values.iloc[0]
                        else:
                            # 对于数值变量，取平均值
                            meta_obs[col] = group_obs.iloc[meta_cell_indices][col].mean()
                    
                    # 添加元细胞信息
                    meta_obs['n_cells_in_meta'] = n_cells_per_meta
                    meta_obs['meta_cell_id'] = f"meta_{total_meta_cells}"
                    meta_obs['original_cells'] = ','.join(map(str, meta_cell_indices))
                    
                    meta_obs_list.append(meta_obs)
                    total_meta_cells += 1
                    
                    # 记录映射
                    meta_cell_mapping.append({
                        'meta_cell_id': meta_obs['meta_cell_id'],
                        'original_indices': meta_cell_indices,
                        'leiden': meta_obs[leiden_col],
                        'sample': meta_obs[sample_col]
                    })
                
                i += 1
            
            # 处理剩余细胞
            remaining_indices = np.where(~selected_mask)[0]
            remaining_count = len(remaining_indices)
            
            if remaining_count >= min_cells_to_keep:
                # 聚合剩余细胞
                if isinstance(group_X, csr_matrix):
                    meta_X = group_X[remaining_indices].mean(axis=0)
                else:
                    meta_X = np.mean(group_X[remaining_indices], axis=0)
                
                meta_X_list.append(meta_X)
                
                # 创建元细胞的观测数据
                meta_obs = {}
                for col in group_obs.columns:
                    if col == 'leiden_sample':
                        continue
                    if group_obs[col].dtype == 'object' or group_obs[col].dtype.name == 'category':
                        values = group_obs.iloc[remaining_indices][col]
                        meta_obs[col] = values.mode()[0] if not values.mode().empty else values.iloc[0]
                    else:
                        meta_obs[col] = group_obs.iloc[remaining_indices][col].mean()
                
                meta_obs['n_cells_in_meta'] = remaining_count
                meta_obs['meta_cell_id'] = f"meta_{total_meta_cells}"
                meta_obs['original_cells'] = ','.join(map(str, remaining_indices))
                
                meta_obs_list.append(meta_obs)
                total_meta_cells += 1
                
                # 记录映射
                meta_cell_mapping.append({
                    'meta_cell_id': meta_obs['meta_cell_id'],
                    'original_indices': remaining_indices,
                    'leiden': meta_obs[leiden_col],
                    'sample': meta_obs[sample_col]
                })
                
                discarded_cells += 0  # 没有丢弃
            elif remaining_count > 0:
                # 丢弃细胞数不足min_cells_to_keep的组
                discarded_cells += remaining_count
        
        elif group_cells_count >= min_cells_to_keep:
            # 对于细胞数在[min_cells_to_keep, n_cells_per_meta)之间的组
            # 将所有细胞聚合成一个元细胞
            meta_cell_indices = list(range(group_cells_count))
            
            # 计算元细胞的表达（平均值）
            if isinstance(group_X, csr_matrix):
                meta_X = group_X[meta_cell_indices].mean(axis=0)
            else:
                meta_X = np.mean(group_X[meta_cell_indices], axis=0)
            
            meta_X_list.append(meta_X)
            
            # 创建元细胞的观测数据
            meta_obs = {}
            for col in group_obs.columns:
                if col == 'leiden_sample':
                    continue
                if group_obs[col].dtype == 'object' or group_obs[col].dtype.name == 'category':
                    values = group_obs.iloc[meta_cell_indices][col]
                    meta_obs[col] = values.mode()[0] if not values.mode().empty else values.iloc[0]
                else:
                    meta_obs[col] = group_obs.iloc[meta_cell_indices][col].mean()
            
            meta_obs['n_cells_in_meta'] = group_cells_count
            meta_obs['meta_cell_id'] = f"meta_{total_meta_cells}"
            meta_obs['original_cells'] = ','.join(map(str, meta_cell_indices))
            
            meta_obs_list.append(meta_obs)
            total_meta_cells += 1
            
            # 记录映射
            meta_cell_mapping.append({
                'meta_cell_id': meta_obs['meta_cell_id'],
                'original_indices': meta_cell_indices,
                'leiden': meta_obs[leiden_col],
                'sample': meta_obs[sample_col]
            })
        else:
            # 丢弃细胞数不足min_cells_to_keep的组
            discarded_cells += group_cells_count
    
    # 删除临时列
    del adata.obs['leiden_sample']
    
    # 创建新的元细胞 AnnData 对象
    print(f"\n聚合完成!")
    print(f"原始细胞数: {adata.n_obs}")
    print(f"生成的元细胞数: {total_meta_cells}")
    print(f"丢弃的细胞数: {discarded_cells}")
    print(f"压缩比例: {adata.n_obs / total_meta_cells:.2f}倍")
    
    # 组合所有元细胞的表达矩阵
    if isinstance(meta_X_list[0], csr_matrix):
        meta_X = vstack(meta_X_list)
    else:
        meta_X = np.vstack(meta_X_list)
    
    # 创建元细胞的观测数据 DataFrame
    meta_obs_df = pd.DataFrame(meta_obs_list)
    meta_obs_df.index = meta_obs_df['meta_cell_id']
    
    # 创建新的 AnnData 对象
    adata_meta = anndata.AnnData(
        X=meta_X,
        obs=meta_obs_df,
        var=meta_var,
        dtype=meta_X.dtype
    )
    
    # 创建映射信息 DataFrame
    meta_info = pd.DataFrame(meta_cell_mapping)
    
    return adata_meta, meta_info


def visualize_meta_cells(adata, adata_meta, meta_info, 
                         leiden_col='leiden', sample_col='sample'):
    """
    可视化元细胞聚合效果
    """
    import matplotlib.pyplot as plt
    
    # 创建对比图
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. 原始细胞分布
    leiden_counts = adata.obs[leiden_col].value_counts().sort_index()
    axes[0, 0].bar(range(len(leiden_counts)), leiden_counts.values)
    axes[0, 0].set_xlabel('Leiden Cluster')
    axes[0, 0].set_ylabel('Number of Cells')
    axes[0, 0].set_title('Original Cells per Leiden Cluster')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. 元细胞分布
    meta_leiden_counts = adata_meta.obs[leiden_col].value_counts().sort_index()
    axes[0, 1].bar(range(len(meta_leiden_counts)), meta_leiden_counts.values)
    axes[0, 1].set_xlabel('Leiden Cluster')
    axes[0, 1].set_ylabel('Number of Meta Cells')
    axes[0, 1].set_title('Meta Cells per Leiden Cluster')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. 压缩比例
    compression_ratios = []
    leiden_clusters = sorted(adata.obs[leiden_col].unique())
    
    for cluster in leiden_clusters:
        orig_count = (adata.obs[leiden_col] == cluster).sum()
        meta_count = (adata_meta.obs[leiden_col] == cluster).sum()
        if meta_count > 0:
            compression_ratios.append(orig_count / meta_count)
        else:
            compression_ratios.append(0)
    
    axes[0, 2].bar(range(len(compression_ratios)), compression_ratios)
    axes[0, 2].set_xlabel('Leiden Cluster')
    axes[0, 2].set_ylabel('Compression Ratio')
    axes[0, 2].set_title('Compression Ratio per Cluster')
    axes[0, 2].tick_params(axis='x', rotation=45)
    
    # 4. 样本分布比较
    sample_counts_orig = adata.obs[sample_col].value_counts().head(10)
    axes[1, 0].bar(range(len(sample_counts_orig)), sample_counts_orig.values)
    axes[1, 0].set_xlabel('Sample')
    axes[1, 0].set_ylabel('Number of Cells')
    axes[1, 0].set_title('Original Cells per Sample(Top 10)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 5. 元细胞样本分布
    sample_counts_meta = adata_meta.obs[sample_col].value_counts().head(10)
    axes[1, 1].bar(range(len(sample_counts_meta)), sample_counts_meta.values)
    axes[1, 1].set_xlabel('Sample')
    axes[1, 1].set_ylabel('Number of Meta Cells') 
    axes[1, 1].set_title('Meta Cells per Sample(Top 10)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # 6. 元细胞大小分布
    meta_sizes = adata_meta.obs['n_cells_in_meta'].value_counts().sort_index()
    axes[1, 2].bar(meta_sizes.index.astype(str), meta_sizes.values)
    axes[1, 2].set_xlabel('Meta Cell Size')
    axes[1, 2].set_ylabel('Number of Meta Cells')
    axes[1, 2].set_title('Distribution of Meta Cell Sizes')
    axes[1, 2].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    
    # 打印详细统计
    print("=" * 60)
    print("元细胞聚合详细统计")
    print("=" * 60)
    # 按leiden统计
    leiden_stats = []
    for cluster in leiden_clusters:
        orig_mask = adata.obs[leiden_col] == cluster
        meta_mask = adata_meta.obs[leiden_col] == cluster
        
        orig_count = orig_mask.sum()
        meta_count = meta_mask.sum()
        
        if meta_count > 0:
            ratio = orig_count / meta_count
            avg_size = adata_meta.obs.loc[meta_mask, 'n_cells_in_meta'].mean()
            
            leiden_stats.append({
                'Leiden': cluster,
                'Original_Cells': orig_count,
                'Meta_Cells': meta_count,
                'Compression_Ratio': f"{ratio:.2f}",
                'Avg_Meta_Size': f"{avg_size:.1f}"
            })
    
    leiden_stats_df = pd.DataFrame(leiden_stats)
    print("\n按Leiden聚类统计:")
    print(leiden_stats_df.to_string(index=False))
    
    # 按样本统计
    sample_stats = []
    top_samples = adata.obs[sample_col].value_counts().head(10).index
    
    for sample in top_samples:
        orig_mask = adata.obs[sample_col] == sample
        meta_mask = adata_meta.obs[sample_col] == sample
        orig_count = orig_mask.sum()
        meta_count = meta_mask.sum()
        
        if meta_count > 0:
            ratio = orig_count / meta_count
            
            sample_stats.append({
                'Sample': sample,
                'Original_Cells': orig_count,
                'Meta_Cells': meta_count,
                'Compression_Ratio': f"{ratio:.2f}"
            })
    
    sample_stats_df = pd.DataFrame(sample_stats)
    print("\n按样本统计 (Top 10):")
    print(sample_stats_df.to_string(index=False))


In [ ]:
########## 这里没有用 adata 的raw的表达数据来产 meta-cell, 做下整合看下
adata_meta, meta_info = create_meta_cells(
        adata, 
        n_cells_per_meta=20, 
        min_cells_to_keep=15,
        leiden_col='leiden', 
        sample_col='sample',
        use_rep='X_pca',
        random_state=42
    )
    

In [ ]:
visualize_meta_cells(adata, adata_meta, meta_info)


In [ ]:
adata.raw[0:100,0:10]


In [ ]:
os.getcwd()
adata_meta.write('adata_meta.h5ad', compression='gzip')
meta_info.to_csv('meta_cell_mapping.csv', index=False)


In [ ]:
adata.raw


In [ ]:
adata.shape

In [ ]:
adata.obs[['leiden','Celltype','Celltype_sub','Celltype_general']].drop_duplicates()


In [ ]:
adata.shape   # (37451, 29037)


In [ ]:
adata.obs[adata.obs['Age_group']=='H']['tissue'].value_counts()


In [ ]:
# sorted(adata.obs['leiden'].unique())


In [ ]:
sc.set_figure_params(figsize=(6, 6), dpi=100) 
sc.pl.umap(adata, color=['leiden','Celltype'],  show=True, ncols =2, wspace=0.3,hspace=0.2 ) 
##### 根据 umap 中的位置把 24,18, 33,23,29 这几个离群的 cluster先去掉, 


In [ ]:
adata = adata[~adata.obs['leiden'].isin(['24','18', '33','23','29','32'])]


In [ ]:
# adata.obs['leiden'].value_counts()
adata


In [ ]:
sc.set_figure_params(figsize=(8, 6), dpi=100) 
sc.pl.umap(adata, color=['leiden','Celltype_general','Celltype'],  show=True, ncols =3, wspace=0.3,hspace=0.2 ) 



In [ ]:
sc.set_figure_params(figsize=(8, 6), dpi=100) 
sc.pl.umap(adata, color=['leiden','Celltype_general','Celltype'],  show=True, ncols =3, wspace=0.3,hspace=0.2 ) 


In [ ]:
########## 然后用去掉离群 cluster 的adata对象做 bbknn, 考虑从 tissue level来去batch, 我之前去 batch 的方法
import harmonypy as hm
cells = adata.shape[0]
# adata = adata.raw.to_adata()
if cells <= 5000:
    n_pcs = 15
    resolution = 0.6
elif cells <= 10000:
    n_pcs = 20
    resolution = 0.8
elif cells <= 40000:
    n_pcs = 30
    resolution = 1
elif cells <= 80000:
    n_pcs = 40
    resolution =1
elif cells <= 150000:
    n_pcs = 50
    resolution = 1
else:
    n_pcs = 75
    resolution = 1


## adata.to_df().min().min()    # len(adata), 其实是gene 的长度, 所以 adata.to_df().min() 返回的是每个gene在所有细胞中的最低表达值
adata.var_names_make_unique()   #### gene name unique
adata.obs_names_make_unique()
sc.pl.highest_expr_genes(adata, n_top=20) 

gsm_counts = adata.obs['tissue'].value_counts()
valid_gsm = gsm_counts[gsm_counts > 2].index
mask = adata.obs['tissue'].isin(valid_gsm)
adata = adata[mask]
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)  ### mito genes
sc.pp.normalize_total(adata, target_sum=1e4)   ### do normalization
sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)  ## 跑完这步会丢失很多gene
adata.raw = adata
# adata = adata[:, adata.var.highly_variable]  ## 只挑选 highly_variable也不对,其实跑完sc.pp.highly_variable_genes, adata的维度就变了
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata,svd_solver="arpack")   ### PCA
sc.external.pp.bbknn(adata,n_pcs=30,neighbors_within_batch=3, batch_key="tissue")    ###  细胞太少了，neighbors变成 1，不然做不了bbknn
# sc.pp.neighbors(adata, n_neighbors=10, n_pcs=n_pcs) #### Find neighborhoods, 用BBKNN做integration，不需要Find Neighbors
sc.tl.umap(adata)
sc.tl.leiden( adata, resolution=resolution,random_state=0,  # flavor="igraph", 
    n_iterations=2, directed=False)  



In [ ]:
sc.pl.umap(adata, color=['res2','tissue','Celltype_general','Celltype'],  show=True, ncols =2, wspace=0.3,hspace=0.2 ) 


In [ ]:
adata.obs['tissue'].value_counts()


In [ ]:
# adata[0:10,0:10].to_df()
adata.obs['tissue'].value_counts()



In [ ]:
gse_counts = adata.obs['sample'].value_counts()
print("原始GSE统计:")
print(gse_counts.head())

# 筛选细胞数大于50的GSE
valid_gses = gse_counts[gse_counts > 10].index.tolist()
print(f"\n细胞数大于50的GSE数量: {len(valid_gses)}")
print(f"细胞数小于等于50的GSE数量: {len(gse_counts) - len(valid_gses)}")

# 创建子集
adata_sub = adata[adata.obs['sample'].isin(valid_gses)].copy()
# adata = adata_sub


In [ ]:
sorted(dir())


In [ ]:
adata = adata_sub


In [ ]:
adata.obs = adata.obs.astype(str)

In [ ]:
(adata.obs.columns)


In [ ]:
adata.obs['tissue'].value_counts()


In [ ]:
adata.shape


In [ ]:
#################### 换一种batch removal的方法，用文文那个试一下
# adata = adata.raw.to_adata()
import datetime
import bbknn
# sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.umap(adata)
# 需要重新矫正批次
print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("Start BBKNN")
bbknn.bbknn(adata, batch_key='tissue', n_pcs=50)
sc.tl.umap(adata)

sc.tl.leiden(adata, resolution=0.8,key_added='res0.8')
sc.tl.leiden(adata, resolution=0.9,key_added='res0.9')
sc.tl.leiden(adata, resolution=1 ,key_added='res1')
sc.tl.leiden(adata, resolution=1.5 ,key_added='res1.5')
sc.tl.leiden(adata, resolution=1.6 ,key_added='res1.6')
sc.tl.leiden(adata, resolution=1.7 ,key_added='res1.7')
sc.tl.leiden(adata, resolution=1.8 ,key_added='res1.8')
sc.tl.leiden(adata, resolution=2 ,key_added='res2')


In [ ]:
# sorted(adata.obs.columns)


In [ ]:
############# 这里是挑选 一个GSE中至少有好几个 sample，并且 sample符合条件的
def select_healthy_samples(adata_list_merge, n_select=5):
    # 存储符合条件的adata对象及其统计信息
    candidates = []
    for adata in adata_list_merge:
        # 检查是否包含Healthy或Normal
        if 'disease_global' not in adata.obs.columns:
            continue
            
        # 检查是否包含Healthy或Normal
        disease_values = adata.obs['disease_global'].unique()
        if not any(d in ['Healthy', 'Normal'] for d in disease_values):
            continue
        
        # 检查是否包含sample列
        if 'sample' not in adata.obs.columns:
            continue
        
        # 获取样本信息
        sample_info = adata.obs['sample']
        
        # 统计每个sample的细胞数
        sample_cell_counts = sample_info.value_counts()
        
        # 筛选出细胞数>=50的sample
        valid_samples = sample_cell_counts[sample_cell_counts >= 50]
        
        # 如果符合条件的sample数量>=5，记录该adata
        if len(valid_samples) >= 5:
            candidates.append({
                'adata': adata,
                'n_samples': len(valid_samples),
                'sample_cell_counts': sample_cell_counts,
                'valid_samples': valid_samples
            })
    
    # 按符合条件的sample数量降序排序
    candidates.sort(key=lambda x: x['n_samples'], reverse=True)
    
    # 返回前n_select个adata对象
    selected_adata_list = [candidate['adata'] for candidate in candidates[:n_select]]
    
    # 打印筛选信息
    print(f"找到 {len(candidates)} 个符合条件的GSE对象")
    print(f"挑选了 {len(selected_adata_list)} 个对象\n")
    
    for i, candidate in enumerate(candidates[:n_select]):
        print(f"候选 {i+1}: 包含 {candidate['n_samples']} 个符合条件的sample (细胞数>=50)")
        print(f"  Sample细胞数统计: {candidate['valid_samples'].head(5).to_dict()}")
        print()
    
    return selected_adata_list

# 使用函数
# selected_adata_list = select_healthy_samples(adata_list_merge, n_select=5)

# # 验证结果
# print(f"最终选择了 {len(selected_adata_list)} 个单细胞对象")
# for i, adata in enumerate(selected_adata_list):
#     print(f"第{i+1}个对象:")
#     print(f"  - disease_global: {adata.obs['disease_global'].unique()}")
#     print(f"  - 总细胞数: {adata.n_obs}")
#     print(f"  - sample数量: {adata.obs['sample'].nunique()}")
#     print(f"  - 细胞数>=50的sample数量: {sum(adata.obs['sample'].value_counts() >= 50)}")
#     print()





In [ ]:
# adata.shape.obs.columns


In [ ]:
adata.obs['tissue'].value_counts()


In [ ]:
sc.pl.umap(adata, color=['Celltype_general'],  show=True, ncols =2, wspace=0.3,hspace=0.2 ) 


In [ ]:
sc.pl.umap(adata, color=['res2','tissue','Celltype_general','Celltype'],  show=True, ncols =2, wspace=0.3,hspace=0.2 ) 


In [ ]:
adata.write('./data/scalex/fibro/human/trajectory/data/FibAtlas_CelltypePan_BatT_47646_Saturn.h5ad')


In [ ]:
adata.shape


In [ ]:
from sklearn.metrics import silhouette_score

# NOTE: BBKNN corrects the graph, not the PCA X_pca directly. 
# Therefore, we evaluate using the UMAP embedding coordinates 
# as a proxy for the manifold structure.

# 1. Batch Entropy / Silhouette (Lower is better for Batch Silhouette if perfectly mixed, 
# but usually we want Silhouette ~ 0 for batch, meaning random mixing)

print("Calculating Silhouette Score for Batch (Tissue)...")
# We subsample for speed if dataset is huge (>50k cells)
idx = np.random.choice(adata.shape[0], size=min(10000, adata.shape[0]), replace=False)
X_umap_sub = adata.obsm['X_umap'][idx]
batch_labels_sub = adata.obs['tissue_system'].iloc[idx]

sil_batch = silhouette_score(X_umap_sub, batch_labels_sub)
print(f"Batch Silhouette (Tissue_system): {sil_batch:.4f} (Closer to 0 is better mixing)")

# 2. Bio Conservation (Leiden)
# High silhouette score means clusters are well separated (good)
cluster_labels_sub = adata.obs['leiden'].iloc[idx]
sil_bio = silhouette_score(X_umap_sub, cluster_labels_sub)
print(f"Bio Silhouette (Leiden): {sil_bio:.4f} (Higher is better separation)")

# --- Visualization of Scores ---
scores = {'Metric': ['Batch Mixing (Tissue)', 'Bio Conservation (Leiden)'],
          'Silhouette Score': [sil_batch, sil_bio]}

import pandas as pd
df_scores = pd.DataFrame(scores)

plt.figure(figsize=(6, 6))

In [ ]:
adata.obs[['leiden','Celltype_general']].drop_duplicates().sort_values('Celltype_general')


In [ ]:
sc.pl.umap(adata, color=['res2','tissue','Celltype_general','Celltype','leiden'],  show=True, ncols =2, wspace=0.1,hspace=0.2 ) 

In [ ]:
# adata.write('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_metacell_gsebatch.h5ad')


In [ ]:
sc.pl.umap(adata, color=['res1','tissue','Celltype_general','Celltype'],  show=True, ncols =2, wspace=0.1,hspace=0.2 ) 


In [ ]:
# sorted(dir())


In [ ]:
adata


In [ ]:
adata.write('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_ds_pan_each2k_bbknn_gsebatch_raw_log1p.h5ad')


In [ ]:
sc.pl.umap(adata, color=['label','tissue','Celltype_general','Celltype'],  show=True, ncols =2, wspace=0.2,hspace=1 ) 


In [ ]:
adata.obs[['leiden','label','Celltype']].drop_duplicates()


In [ ]:
adata.obs[['leiden','label','Celltype']].drop_duplicates()


In [ ]:
# adata.obs['tissue_final'].value_counts()


In [ ]:
sc.pl.umap(adata, color=['leiden','Celltype_general','Celltype'],  show=True, ncols =3, wspace=0.2,hspace=0.2 ) 


In [ ]:
#################### 下面开始用整合后的数据做 stream 分析
cell = pd.DataFrame({'Cell':adata.obs['Celltype_general'].to_list()})
# cell.to_csv('./data/scalex/fibro/human/stream_result/subtype_only/'+subtype+'/'+subtype+'_ds.tsv',index=False, sep='\t')
cell.to_csv('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/celltype_general_ds_each2k_tissuebatch_raw_nolog1p.tsv',index=False, sep='\t',header=None)


In [ ]:
#################### 下面开始用整合后的数据做 stream 分析
cell = pd.DataFrame({'Cell':adata.obs['Celltype'].to_list()})
# cell.to_csv('./data/scalex/fibro/human/stream_result/subtype_only/'+subtype+'/'+subtype+'_ds.tsv',index=False, sep='\t')
cell.to_csv('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/celltype_ds_each2k_tissuebatch_raw_nolog1p.tsv',index=False, sep='\t',header=None)


In [ ]:
adata.obs['res2'].unique()


In [ ]:
adata = sc.read_h5ad('./data/scalex/endo/human/trajectory/data/EndoAtlas_CelltypePan_BatT_19011.h5ad')


In [ ]:
# st.add_cell_labels(adata,file_name='./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/celltype_general_ds_each2k_tissuebatch_new.tsv')
st.set_workdir(adata,'./data/scalex/endo/human/trajectory/cellrank/bbknn')


In [ ]:
adata.shape


In [ ]:
adata.obs['res2'] = adata.obs['res2'].astype(str)
adata.obs['label'] = adata.obs['res2'] 
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")
sc.pl.umap(adata, color=["label"],show=True)


In [ ]:
def se_on_bbknn_neighbors(adata, n_components=3, random_state=42):
    A = adata.obsp["connectivities"]   # bbknn 生成的图
    se = SpectralEmbedding(
        n_components=n_components,
        affinity="precomputed",   # 关键 ✨
        random_state=random_state
    )
    X_se = se.fit_transform(A)
    adata.obsm["X_se"] = X_se
    adata.obsm["X_dr"] = X_se
    adata.uns["trans_se"] = se
    return(adata)
    

In [ ]:
adata.obs['Celltype_general'].value_counts()


In [ ]:
adata.obs['Celltype_general'].value_counts()


In [ ]:
# adata.write('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_ds_each1k_bbknn_tissuebatch.h5ad')




In [ ]:
adata = sc.read_h5ad('./data/scalex/fibro/human/trajectory/data/FibAtlas_CelltypePan_BatT_47679_Saturn.h5ad')
adata.shape


In [ ]:
adata=se_on_bbknn_neighbors(adata, n_components=3, random_state=42)


In [ ]:
adata_endo = adata


In [ ]:
adata.obs["label"] = adata.obs["label"].astype(str)
n_neighbors=100
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 3), facecolor="white")
st.plot_dimension_reduction(adata,color=['label'],show_graph=False,show_text=False)
st.plot_visualization_2D(adata,method='umap',n_neighbors=n_neighbors,color=['label'],use_precomputed=False)
# plt.savefig(directory+'umap_2D.png', dpi=300, bbox_inches='tight', facecolor='white')
st.elastic_principal_graph(adata,epg_alpha=0.02,epg_mu=0.1,epg_lambda=0.02)
st.plot_dimension_reduction(adata,color=['label'],n_components=2,show_graph=True,show_text=False)
st.plot_branches(adata,show_text=True)


In [ ]:

endo_subtype = ['LEC','VenEC'] #   ['ArtEC','CapEC']  # ['MCs','EndoMT']     ['LEC']   ['VenEC']
adata = adata_endo[adata_endo.obs['Celltype_general'].isin(endo_subtype)]


In [ ]:
################### Saturn ，jupyter 的, Endothelial的 stream 分析
st.set_figure_params(dpi=80,style='white',figsize=[5.4,4.8], rc={'image.cmap': 'viridis'})
n_clusters=10
n_neighbors=100
cell_num =''
adata.obs['Celltype_general'] = adata.obs['Celltype_general'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
adata.obs['label'] = adata.obs['Celltype']
directory = './data/scalex/endo/human/trajectory/stream_result/'
st.set_workdir(adata,directory)
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")
sc.pl.umap(adata, color=["label"],show=True)
# plt.savefig(directory+cell_num+'_umap.png', dpi=300, bbox_inches='tight', facecolor='white')
st.cal_qc(adata,assay='rna')
# st.filter_features(adata,min_n_cells = 5)
# st.filter_cells(adata,min_n_features= 100)
st.select_variable_genes(adata,loess_frac=0.01)
# plt.savefig(directory+cell_num+'_filter_genes_0.01.png',dpi=300, bbox_inches='tight', facecolor='white')
# plt.savefig(directory+cell_num+'_filter_genes_0.1.png',dpi=300, bbox_inches='tight', facecolor='white')
adata.obs["label"] = adata.obs["label"].astype(str)
st.dimension_reduction(adata,method='pca',feature='var_genes', n_neighbors=50,n_components=50,n_jobs=2)   
st.plot_dimension_reduction(adata,color=['label'], show_graph=False,show_text=False,fig_legend_ncol=1)
# plt.savefig(directory+cell_num+'_dimension_reduction_3D.png', dpi=300,   bbox_inches='tight',  facecolor='white')  
st.plot_visualization_2D(adata,method='umap',n_neighbors=n_neighbors,color=['label'],use_precomputed=False)
# plt.savefig(directory+cell_num+'_umap_2D.png', dpi=300, bbox_inches='tight', facecolor='white')
st.seed_elastic_principal_graph(adata,n_clusters=7,use_vis=True) #  5个大分支
st.plot_dimension_reduction(adata,color=['label'],n_components=2,show_graph=True,show_text=True,fig_ncol=3)
# plt.savefig(directory+cell_num+'_label_2D.png', dpi=300, bbox_inches='tight', facecolor='white')
st.plot_branches(adata,show_text=True)


In [ ]:
adata


In [ ]:
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 3), facecolor="white")

# st.elastic_principal_graph(adata,epg_alpha=0.02,epg_mu=0.1,epg_lambda=0.02)
st.plot_dimension_reduction(adata,color=['label'],n_components=2,show_graph=True,show_text=False)
st.plot_branches(adata,show_text=True)
# plt.savefig(directory+cell_num+'_branch_line.png', dpi=300, bbox_inches='tight', facecolor='white')

st.plot_stream_sc(adata,root='S1',color=['label'],
                  dist_scale=0.5,show_graph=True,show_text=True)
plt.savefig(directory+cell_num+'_'+ 'stream_sc_s1.png', dpi=300, bbox_inches='tight', facecolor='white')
st.plot_stream_sc(adata,root='S0',color=['label'],
                  dist_scale=0.5,show_graph=True,show_text=True)
plt.savefig(directory+cell_num+'_'+ 'stream_sc_s0.png', dpi=300, bbox_inches='tight', facecolor='white')
st.plot_stream_sc(adata,root='S2',color=['label'],
                  dist_scale=0.5,show_graph=True,show_text=True)
st.plot_stream_sc(adata,root='S3',color=['label'],
                  dist_scale=0.5,show_graph=True,show_text=True)

st.plot_stream(adata,root='S0',color=['label'])
#plt.savefig(directory+cell_num+'_'+ 'stream_st_s0.png', dpi=300, bbox_inches='tight', facecolor='white')
st.plot_stream(adata,root='S1',color=['label'])
#plt.savefig(directory+cell_num+'_'+ 'stream_st_s1.png', dpi=300, bbox_inches='tight', facecolor='white')
st.plot_stream(adata,root='S2',color=['label'])
#plt.savefig(directory+cell_num+'_'+ 'stream_st_s2.png', dpi=300, bbox_inches='tight', facecolor='white')
st.plot_stream(adata,root='S3',color=['label'])
# st.plot_stream(adata,root='S4',color=['label'])
# st.plot_stream(adata,root='S5',color=['label'])


In [ ]:
adata


In [ ]:
adata.obs.columns


In [ ]:
######## calculate the entropy based on gene (Gene Shannon entropy )
# adata.obs['Celltype_general'].value_counts()
import scanpy as sc
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

def calculate_gene_entropy(adata, layer=None, use_raw=False):
    # 获取表达矩阵
    if use_raw and adata.raw is not None:
        X = adata.raw.X
    elif layer is not None and layer in adata.layers:
        X = adata.layers[layer]
    else:
        X = adata.X
    # 转换为稠密矩阵（如果是稀疏矩阵）
    if hasattr(X, 'toarray'):
        X = X.toarray()
    # 方法1：计算每个细胞的entropy（基因分布的均匀性）
    cell_entropy = []
    for i in range(X.shape[0]):
        # 获取该细胞的表达向量，并归一化为概率分布
        exp_vec = X[i, :]
        # 添加小值避免log(0)
        exp_vec = exp_vec + 1e-10
        # 归一化
        p = exp_vec / exp_vec.sum()
        # 计算Shannon entropy
        entropy = -np.sum(p * np.log2(p))
        cell_entropy.append(entropy)
    adata.obs['cell_entropy'] = cell_entropy
    # 方法2：计算每个基因的entropy（在细胞间的表达异质性）
    gene_entropy = []
    for j in range(X.shape[1]):
        exp_vec = X[:, j]
        exp_vec = exp_vec + 1e-10
        p = exp_vec / exp_vec.sum()
        entropy = -np.sum(p * np.log2(p))
        gene_entropy.append(entropy)
    adata.var['gene_entropy'] = gene_entropy
    return adata


adata = calculate_gene_entropy(adata)


In [ ]:
# 步骤1：完整数据预处理（补全缺失代码，确保数据可用）
# --------------------------
age_gsm_df = Fib_meta_data[["Age_group", "GSM"]].copy()
age_gsm_df.rename(columns={'Age_group': 'age_group'}, inplace=True)

# 统计细胞数和原始GSM数
age_cell_count = age_gsm_df.groupby("age_group").size().reset_index(name="cell_count")
age_gsm_count = age_gsm_df.groupby("age_group")["GSM"].nunique().reset_index(name="gsm_count")

# 合并数据并处理
age_stats = pd.merge(age_cell_count, age_gsm_count, on="age_group", how="outer")
age_stats = age_stats.dropna(subset=["cell_count", "gsm_count"]).reset_index(drop=True)
age_stats["cell_count"] = age_stats["cell_count"].astype(int)
age_stats["gsm_count"] = age_stats["gsm_count"].astype(int)

# 设置age_group顺序（与实际分组一致）
custom_age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
age_stats = age_stats[age_stats["age_group"].isin(custom_age_order)].copy()
age_stats["age_group"] = pd.Categorical(age_stats["age_group"], categories=custom_age_order, ordered=True)
age_stats = age_stats.sort_values("age_group").reset_index(drop=True)

# --------------------------
# 核心1：计算放大500倍的GSM负向高度（柱子高度=实际GSM数×500，负向显示）
# --------------------------
zoom_factor = 500  # 固定放大500倍
age_stats["gsm_count_neg_zoom"] = -age_stats["gsm_count"] * zoom_factor  # 放大后的负向高度

# 打印统计结果（确认实际GSM数与放大后高度）
print("实际GSM数与放大后柱子高度：")
print(age_stats[["age_group", "gsm_count", "gsm_count_neg_zoom"]])
print(f"\n放大倍数：{zoom_factor}倍（GSM柱子高度=实际GSM数×{zoom_factor}）")


# --------------------------
# 步骤2：绘制柱状图（核心2：自定义y轴负向刻度标签）
# --------------------------
if not age_stats.empty:
    plt.figure(figsize=(10, 6))
    # 定义颜色
    cell_color = '#1F77B4'  # colors = ['#1F77B4', '#FF7F0E']
    gsm_color = '#FF7F0E'
    # 1. 绘制细胞数（向上，原始高度，不变）
    bars_cell = plt.bar(
        x=age_stats["age_group"],
        height=age_stats["cell_count"],
        color=cell_color,
        alpha=0.8,
        label="Cell Number"
    )
    # 2. 绘制GSM数（向下，放大500倍的高度）
    bars_gsm = plt.bar(
        x=age_stats["age_group"],
        height=age_stats["gsm_count_neg_zoom"],  # 用放大后的高度
        color=gsm_color,
        alpha=0.8,
        label="Sample Number"
    )
    ax = plt.gca()  # 获取当前坐标轴对象
    y_ticks = ax.get_yticks()  # 所有y轴刻度值（如[-6500, -3250, 0, 29027, 58055, ...]）
    # 自定义刻度标签：负向刻度÷-500（还原为实际GSM数），正向刻度保持不变
    custom_y_labels = []
    for tick in y_ticks:
        if tick < 0:  # 负向刻度（对应GSM）：刻度值 ÷ (-zoom_factor) = 实际GSM数
            actual_gsm = int(tick / (-zoom_factor))  # 如-6500 ÷ -500 = 13
            custom_y_labels.append(str(actual_gsm))  # 显示实际GSM数（如"13"）
        elif tick == 0:  # 0刻度：保持"0"
            custom_y_labels.append("0")
        else:  # 正向刻度（对应细胞数）：保持原始数值（可加千分位）
            custom_y_labels.append(f"{int(tick):,}")  # 如"29,027"
    # 应用自定义刻度标签（关键！让y轴负向显示实际GSM数）
    ax.set_yticklabels(custom_y_labels, fontsize=9)
    # --------------------------
    # 标注与美化（标注均显示实际数量，无放大）
    # --------------------------
    # 细胞数标注（显示实际数量，千分位）
    for bar, count in zip(bars_cell, age_stats["cell_count"]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + age_stats["cell_count"].max() * 0.01,
            s=f"{count:,}",
            ha="center", va="bottom",
            fontsize=9,
            fontweight="bold"
        )
    # GSM标注（显示实际数量，无需放大）
    for bar, count in zip(bars_gsm, age_stats["gsm_count"]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() - (age_stats["gsm_count"].max() * zoom_factor) * 0.02,  # 基于放大后高度的偏移
            s=f"{count}",  # 显示实际GSM数（如13）
            ha="center", va="top",
            fontsize=9,
            fontweight="bold",
            color=gsm_color
        )
    plt.xlabel("Age Group", fontsize=12, labelpad=10)
    plt.ylabel("Count", 
               fontsize=11, labelpad=10)
    plt.title(f"Fibro distribution of AgeGroup", 
              fontsize=14, pad=20)
    # 图例和0刻度线（加粗0刻度，区分上下区域）
    plt.legend(loc="upper center", fontsize=10, bbox_to_anchor=(0.5, -0.08), ncol=2)
    plt.axhline(y=0, color="black", linestyle="-", linewidth=2)
    # 调整布局，避免标签截断
    plt.tight_layout
    plt.savefig('external_inputs/Fibro_cellsample_agegroup.png', dpi=300, bbox_inches='tight')
    plt.savefig('external_inputs/Fibro_cellsample_agegroup.pdf', format='pdf', bbox_inches='tight')
    plt.show()
else:
    print("无可用数据用于绘图，请检查custom_age_order是否与age_group匹配！")
    

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import matplotlib.ticker as ticker
from matplotlib.patches import Patch

# 设置字体和绘图样式
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
rcParams['pdf.fonttype'] = 42  # 确保PDF中文字体可编辑

# 1. 数据合并和处理
def prepare_combined_data(Fib_meta_data, meta_data):
    """
    合并Fibroblast和Endothelial数据，并准备可视化数据
    """
    # 提取Fibroblast数据（从数据框A）
    fib_data = Fib_meta_data[['Lineage', 'Age_group', 'GSM']].copy()
    
    # 提取Endothelial数据（从数据框B）
    endo_data = meta_data[['Lineage', 'Age_group', 'GSM']].copy()
    
    # 按行合并两个数据框
    age_gsm_df = pd.concat([fib_data, endo_data], ignore_index=True)
    
    print(f"合并后数据形状: {age_gsm_df.shape}")
    print(f"Lineage分布:")
    print(age_gsm_df['Lineage'].value_counts())
    
    return age_gsm_df

# 2. 数据处理和统计
def prepare_visualization_data(age_gsm_df):
    """
    准备可视化所需的数据统计
    """
    # 规范列名
    age_gsm_df = age_gsm_df.copy()
    age_gsm_df.rename(columns={'Age_group': 'age_group'}, inplace=True)
    
    # 定义年龄组顺序
    custom_age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    
    # 确保只包含指定年龄组
    age_gsm_df = age_gsm_df[age_gsm_df['age_group'].isin(custom_age_order)].copy()
    age_gsm_df['age_group'] = pd.Categorical(
        age_gsm_df['age_group'], 
        categories=custom_age_order, 
        ordered=True
    )
    
    # 统计每个年龄组、每个Lineage的细胞数
    cell_stats = age_gsm_df.groupby(['age_group', 'Lineage']).size().reset_index(name='cell_count')
    
    # 统计每个年龄组、每个Lineage的样本数（唯一GSM数）
    sample_stats = age_gsm_df.groupby(['age_group', 'Lineage'])['GSM'].nunique().reset_index(name='sample_count')
    
    # 合并统计数据
    stats_df = pd.merge(cell_stats, sample_stats, on=['age_group', 'Lineage'], how='outer')
    
    # 确保数据类型正确
    stats_df['cell_count'] = stats_df['cell_count'].fillna(0).astype(int)
    stats_df['sample_count'] = stats_df['sample_count'].fillna(0).astype(int)
    
    # 排序
    stats_df = stats_df.sort_values(['age_group', 'Lineage']).reset_index(drop=True)
    
    # 打印统计摘要
    print("\n统计摘要:")
    print("=" * 60)
    for lineage in stats_df['Lineage'].unique():
        lineage_data = stats_df[stats_df['Lineage'] == lineage]
        total_cells = lineage_data['cell_count'].sum()
        total_samples = lineage_data['sample_count'].sum()
        print(f"{lineage}: 总细胞数 = {total_cells:,}, 总样本数 = {total_samples:,}")
    
    return stats_df, custom_age_order

# 3. 创建对称柱状图（统一颜色方案）
def create_symmetric_barplot_with_lineages(stats_df, custom_age_order, zoom_factor=500, 
                                           figsize=(14, 10), 
                                           save_path='dual_lineage_age_distribution'):
    """
    创建包含两种Lineage的对称柱状图
    
    参数:
    stats_df: 统计数据框
    custom_age_order: 年龄组顺序
    zoom_factor: 样本数放大倍数
    figsize: 图形大小
    save_path: 保存路径前缀
    """
    
    if stats_df.empty:
        print("无可用数据用于绘图！")
        return None, None, None
    
    # 固定颜色方案
    CELL_COLOR = '#1F77B4'  # 细胞数 - 蓝色
    SAMPLE_COLOR = '#FF7F0E'  # 样本数 - 橙色
    
    # 透明度设置 - 用于区分不同Lineage
    # Fibroblast: 较高透明度，Endothelial: 较低透明度
    ALPHAS = {
        'Fibroblast': 0.85,
        'Endothelial': 0.55
    }
    
    # 创建图形和轴
    fig, ax = plt.subplots(figsize=figsize)
    
    # 获取唯一的年龄组和Lineage
    age_groups = stats_df['age_group'].unique()
    lineages = sorted(stats_df['Lineage'].unique())  # 排序确保一致
    
    # 设置条形位置
    n_age_groups = len(age_groups)
    n_lineages = len(lineages)
    
    # 设置条形宽度和位置偏移
    total_width = 0.8  # 每组条形的总宽度
    bar_width = total_width / n_lineages  # 每个条形的宽度
    
    x_positions = np.arange(n_age_groups)
    
    # 计算样本数放大后的负向高度
    stats_df['sample_count_neg_zoom'] = -stats_df['sample_count'] * zoom_factor
    
    print(f"\n使用放大倍数: {zoom_factor}倍")
    print(f"样本数柱子高度 = 实际样本数 × {zoom_factor}")
    
    # 绘制条形图
    bars_cell = {}  # 存储细胞数条形
    bars_sample = {}  # 存储样本数条形
    
    for i, lineage in enumerate(lineages):
        # 提取当前Lineage的数据
        lineage_data = stats_df[stats_df['Lineage'] == lineage].copy()
        
        # 确保数据顺序正确
        lineage_data = lineage_data.set_index('age_group').reindex(custom_age_order).reset_index()
        lineage_data = lineage_data.fillna({'cell_count': 0, 'sample_count': 0, 
                                            'sample_count_neg_zoom': 0})
        
        # 计算条形偏移位置
        offset = (i - (n_lineages - 1) / 2) * bar_width
        
        # 绘制细胞数条形（x轴上方）- 统一使用蓝色
        bars_cell[lineage] = ax.bar(
            x_positions + offset,
            lineage_data['cell_count'].values,
            width=bar_width * 0.9,  # 稍微窄一点，避免重叠
            color=CELL_COLOR,
            alpha=ALPHAS[lineage],
            edgecolor='black',
            linewidth=1.2,
            label=f'{lineage} - Cell Number'
        )
        
        # 绘制样本数条形（x轴下方，放大后）- 统一使用橙色
        bars_sample[lineage] = ax.bar(
            x_positions + offset,
            lineage_data['sample_count_neg_zoom'].values,
            width=bar_width * 0.9,
            color=SAMPLE_COLOR,
            alpha=ALPHAS[lineage],
            edgecolor='black',
            linewidth=1.2,
            label=f'{lineage} - Sample Number'
        )
    
    # 设置x轴
    ax.set_xlabel('Age Group', fontsize=16, fontweight='bold', labelpad=15)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(custom_age_order, fontsize=14, fontweight='bold')
    
    # 设置y轴范围和刻度
    # 计算y轴范围
    max_cell = stats_df['cell_count'].max()
    max_sample_scaled = abs(stats_df['sample_count_neg_zoom']).max()
    
    # 调整y轴范围，使上下比例均衡
    y_max = max_cell * 1.15
    y_min = -max_sample_scaled * 1.15
    
    ax.set_ylim(y_min, y_max)
    ax.set_ylabel('Count', fontsize=15, fontweight='bold', labelpad=15)
    
    # 自定义y轴刻度标签
    def custom_formatter(value, pos):
        if value < 0:
            # 负值区域：显示实际样本数（去掉放大倍数）
            actual_value = abs(value) / zoom_factor
            if actual_value >= 1000:
                return f'{actual_value/1000:.0f}k'
            else:
                return f'{actual_value:.0f}'
        elif value > 0:
            # 正值区域：显示细胞数
            if value >= 1000000:
                return f'{value/1000000:.1f}M'
            elif value >= 1000:
                return f'{value/1000:.0f}k'
            else:
                return f'{value:.0f}'
        else:
            return '0'
    
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
    ax.tick_params(axis='y', labelsize=12)
    
    # 添加粗的x轴参考线
    ax.axhline(y=0, color='black', linewidth=2.5, alpha=0.9, zorder=0)
    
    # 添加网格线
    ax.grid(True, axis='y', alpha=0.2, linestyle='--', linewidth=0.8, zorder=0)
    
    # 添加条形数值标签
    def add_value_labels(bars, ax, values_df, is_sample=False, zoom_factor=1):
        """添加条形数值标签"""
        for bar_group, lineage in zip(bars.values(), bars.keys()):
            # 获取当前Lineage的数据
            lineage_data = values_df[values_df['Lineage'] == lineage].copy()
            lineage_data = lineage_data.set_index('age_group').reindex(custom_age_order).reset_index()
            
            for i, bar in enumerate(bar_group):
                height = bar.get_height()
                
                if is_sample:
                    # 样本数标签（显示实际值）
                    actual_value = lineage_data.loc[i, 'sample_count']
                    label = f'{actual_value:.0f}'
                    
                    if height < 0:
                        # 负值条形（下方）
                        va = 'top'
                        offset = -8
                    else:
                        va = 'bottom'
                        offset = 5
                else:
                    # 细胞数标签
                    value = lineage_data.loc[i, 'cell_count']
                    if value >= 1000000:
                        label = f'{value/1000000:.1f}M'
                    elif value >= 1000:
                        label = f'{value/1000:.0f}k'
                    else:
                        label = f'{value:.0f}'
                    
                    if height > 0:
                        # 正值条形（上方）
                        va = 'bottom'
                        offset = 5
                    else:
                        va = 'top'
                        offset = -8
                
                # 添加标签
                ax.annotate(
                    label,
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, offset),
                    textcoords='offset points',
                    ha='center', va=va,
                    fontsize=10,
                    fontweight='bold' if not is_sample else 'normal',
                    color='black',
                    bbox=dict(
                        boxstyle='round,pad=0.2',
                        facecolor='white',
                        edgecolor='gray',
                        alpha=0.7
                    )
                )
    
    # 添加细胞数值标签
    add_value_labels(bars_cell, ax, stats_df, is_sample=False)
    
    # 添加样本数值标签
    add_value_labels(bars_sample, ax, stats_df, is_sample=True, zoom_factor=zoom_factor)
    
    # 创建自定义图例
    # 创建图例句柄
    legend_handles = []
    legend_labels = []
    
    # 1. 添加Cell Number和Sample Number方向说明
    legend_handles.append(Patch(facecolor=CELL_COLOR, alpha=0.7, edgecolor='black'))
    legend_labels.append('↑ Cell Number')
    
    legend_handles.append(Patch(facecolor=SAMPLE_COLOR, alpha=0.7, edgecolor='black'))
    legend_labels.append('↓ Sample Number')
    
    # 2. 添加Lineage透明度区分说明
    # Fibroblast (较高透明度)
    fib_patch_cell = Patch(facecolor=CELL_COLOR, alpha=ALPHAS['Fibroblast'], edgecolor='black')
    fib_patch_sample = Patch(facecolor=SAMPLE_COLOR, alpha=ALPHAS['Fibroblast'], edgecolor='black')
    
    # Endothelial (较低透明度)
    endo_patch_cell = Patch(facecolor=CELL_COLOR, alpha=ALPHAS['Endothelial'], edgecolor='black')
    endo_patch_sample = Patch(facecolor=SAMPLE_COLOR, alpha=ALPHAS['Endothelial'], edgecolor='black')
    
    # 添加Lineage图例
    legend_handles.append(fib_patch_cell)
    legend_labels.append('Fibroblast')
    
    legend_handles.append(endo_patch_cell)
    legend_labels.append('Endothelial')
    
    # 创建图例
    legend = ax.legend(legend_handles, legend_labels,
                      loc='upper center',
                      bbox_to_anchor=(0.5, 1.05),  # 将图例放在图表上方
                      ncol=2,  # 两列布局
                      fontsize=12,
                      frameon=True,
                      framealpha=0.95,
                      edgecolor='black',
                      title='Legend',
                      title_fontsize=13)
    
    # 设置标题
    cell_total = stats_df['cell_count'].sum()
    sample_total = stats_df['sample_count'].sum()
    
    title = f'Dual Lineage Distribution of Age Groups\n' \
            f'(Total Cells: {cell_total:,}, Total Samples: {sample_total:,}, ' \
            f'Sample Zoom: {zoom_factor}×)'
    
    ax.set_title(title, fontsize=18, fontweight='bold', pad=25)
    
    # 添加颜色说明
    color_info = f'Colors: Cell Number = {CELL_COLOR}, Sample Number = {SAMPLE_COLOR}\n' \
                 f'Alpha: Fibroblast = {ALPHAS["Fibroblast"]}, Endothelial = {ALPHAS["Endothelial"]}'
    
    ax.text(0.02, 0.98, color_info,
            transform=ax.transAxes,
            fontsize=10,
            fontstyle='italic',
            color='gray',
            verticalalignment='top',
            horizontalalignment='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))
    
    # 添加放大倍数说明
    ax.text(0.98, 0.02, f'Sample bars zoomed {zoom_factor}×',
            transform=ax.transAxes,
            fontsize=11,
            fontstyle='italic',
            color='gray',
            verticalalignment='bottom',
            horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))
    
    # 调整布局
    plt.tight_layout()
    
    # 保存图形
    plt.savefig(f'{save_path}.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(f'{save_path}.pdf', format='pdf', bbox_inches='tight', facecolor='white')
    
    print(f"\n图形已保存为: {save_path}.png 和 {save_path}.pdf")
    
    return fig, ax, stats_df

# 4. 主函数
def visualize_dual_lineage_age_groups(Fib_meta_data, meta_data, zoom_factor=500, 
                                      save_path='dual_lineage_age_distribution'):
    """
    主函数：执行完整的双Lineage可视化流程
    
    参数:
    Fib_meta_data: Fibroblast数据框
    meta_data: Endothelial数据框
    zoom_factor: 样本数放大倍数
    save_path: 保存路径前缀
    """
    
    print("=" * 70)
    print("开始处理双Lineage年龄组分布数据")
    print("=" * 70)
    
    # 1. 合并数据
    print("\n1. 合并Fibroblast和Endothelial数据...")
    age_gsm_df = prepare_combined_data(Fib_meta_data, meta_data)
    
    # 2. 准备可视化数据
    print("\n2. 准备可视化统计数据...")
    stats_df, custom_age_order = prepare_visualization_data(age_gsm_df)
    
    # 3. 创建可视化
    print("\n3. 创建双Lineage对称柱状图...")
    fig, ax, stats_df = create_symmetric_barplot_with_lineages(
        stats_df, 
        custom_age_order, 
        zoom_factor=zoom_factor,
        save_path=save_path
    )
    
    # 4. 显示详细统计
    print("\n4. 详细统计:")
    print("=" * 70)
    print(f"{'Age Group':<10} {'Lineage':<15} {'Cells':<15} {'Samples':<10}")
    print("-" * 70)
    
    for age_group in custom_age_order:
        for lineage in ['Fibroblast', 'Endothelial']:
            subset = stats_df[(stats_df['age_group'] == age_group) & 
                             (stats_df['Lineage'] == lineage)]
            
            if not subset.empty:
                cells = subset.iloc[0]['cell_count']
                samples = subset.iloc[0]['sample_count']
                
                cells_display = f"{cells:,}"
                if cells >= 1000000:
                    cells_display = f"{cells/1000000:.1f}M"
                elif cells >= 1000:
                    cells_display = f"{cells/1000:.0f}k"
                
                print(f"{age_group:<10} {lineage:<15} {cells_display:<15} {samples:<10}")
    
    print("=" * 70)
    
    # 显示图形
    plt.show()
    
    return fig, stats_df, age_gsm_df

# 5. 测试数据生成函数
def create_test_data():
    """
    创建测试数据，模拟您的数据结构
    """
    np.random.seed(42)  # 设置随机种子以确保可重复性
    
    # 模拟年龄组
    custom_age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    
    # 为Fibroblast创建测试数据
    fib_data = []
    for age_group in custom_age_order:
        # 每个年龄组的样本数
        n_samples = np.random.randint(3, 10)
        
        for sample_idx in range(n_samples):
            gsm = f"GSM{np.random.randint(1000000, 9999999)}"
            # 每个样本的细胞数
            n_cells = np.random.randint(50, 200)
            
            for cell_idx in range(n_cells):
                fib_data.append({
                    'Lineage': 'Fibroblast',
                    'Age_group': age_group,
                    'GSM': gsm
                })
    
    # 为Endothelial创建测试数据
    endo_data = []
    for age_group in custom_age_order:
        # Endothelial通常样本数较少
        n_samples = np.random.randint(2, 8)
        
        for sample_idx in range(n_samples):
            gsm = f"GSM{np.random.randint(1000000, 9999999)}"
            # Endothelial细胞数通常较少
            n_cells = np.random.randint(30, 150)
            
            for cell_idx in range(n_cells):
                endo_data.append({
                    'Lineage': 'Endothelial',
                    'Age_group': age_group,
                    'GSM': gsm
                })
    
    # 创建数据框
    fib_df = pd.DataFrame(fib_data)
    endo_df = pd.DataFrame(endo_data) 
    return fib_df, endo_df



In [ ]:
import random
candidate = ["Ke Tang","Hailin Wei","Zhaoyang Liu","Zhonghua Dong","Wenwen Shao","Yongyan Wang","Pengpeng Wu","Xinwei Zheng","Leyi Zhang","Zijia Li","Yazi Li","Xiantong Jiang",
             "Ke Tang","Zhaoyang Liu","Zhonghua Dong","Yazi Li" ]



In [ ]:
import random
candidate = ["Ke Tang","Hailin Wei","Zhaoyang Liu","Zhonghua Dong","Wenwen Shao","Yongyan Wang","Pengpeng Wu","Xinwei Zheng","Leyi Zhang","Zijia Li","Yazi Li","Xiantong Jiang" ]
random.sample(candidate,5)


In [ ]:
# 使用您自己的数据
fig, stats_df, combined_df = visualize_dual_lineage_age_groups(
    Fib_meta_data, 
    meta_data,
    zoom_factor=500,  # 样本数放大倍数
    save_path='external_inputs/dual_lineage_distribution'  # 保存路径
)


In [ ]:
################## scRNA-seq ICB data filtering  ################
import os 
import re 
os.chdir('/fse/home/wangyongyan/Project/parser_code/result/2026_01_ICB_wyt')

file_paths = os.listdir('/fse/home/wangyongyan/Project/parser_code/result/2026_01_ICB_wyt')
file_paths


In [ ]:
treatment_meta_data = pd.read_csv('gse_meta_rectify.csv',sep=',')
treatment_meta_data.head(5)

In [ ]:
icb_keywords = {
    # 主要治疗类型
    "immune_checkpoint_blockade": [
        "immune checkpoint blockade", "ICB", "checkpoint blockade",
        "immune checkpoint inhibitor", "checkpoint inhibitor therapy"
    ],
    # PD-1/PD-L1轴
    "pd1_pdl1": [
        "PD-1", "programmed cell death protein 1", "CD279",
        "PD-L1", "programmed death ligand 1", "CD274", "B7-H1",
        "PD-L2", "programmed death ligand 2", "CD273", "B7-DC",
        "anti-PD-1", "anti-PD-L1", "PD-1 blockade", "PD-L1 blockade",
        "nivolumab", "Opdivo", "pembrolizumab", "Keytruda",
        "atezolizumab", "Tecentriq", "durvalumab", "Imfinzi",
        "avelumab", "Bavencio", "cemiplimab", "Libtayo"
    ],
    # CTLA-4轴
    "ctla4": [
        "CTLA-4", "cytotoxic T-lymphocyte-associated protein 4", 
        "anti-CTLA-4", "CTLA-4 blockade",
        "ipilimumab", "Yervoy", "tremelimumab", "Imjudo"
    ],
    # 其他检查点分子
    "other_checkpoints": [
        "LAG-3", "lymphocyte activation gene 3", 
        "TIM-3", "T cell immunoglobulin mucin 3", 
        "TIGIT", "T cell immunoreceptor with Ig and ITIM domains",
        "VISTA", "V-domain Ig suppressor of T cell activation",
        "B7-H3",
        "IDO1", "indoleamine 2,3-dioxygenase 1",
        "KIR", "killer cell immunoglobulin-like receptor",
        "Tactile"
    ],
    # 联合疗法
    "combination_therapy": [
        "combination immunotherapy", "dual checkpoint blockade",
        "PD-1/CTLA-4 combination", "nivolumab+ipilimumab",
        "pembrolizumab+ipilimumab", "ICB combination",
        "ICB plus chemotherapy", "ICB plus targeted therapy",
        "ICB plus radiotherapy"
    ],
    # 治疗响应相关
    "therapy_response": [
        "responder", "non-responder", "response to ICB",
        "durable clinical benefit", "DCB",
        "progressive disease", "PD", "stable disease", "SD",
        "partial response", "PR", "complete response", "CR",
        "objective response rate", "ORR",
        "progression-free survival", 
        "overall survival", "OS",
        "hyperprogression", "acquired resistance",
        "primary resistance", "adaptive resistance"
    ],
    # 生物标志物
    "biomarkers": [
        "tumor mutational burden", "TMB",
        "microsatellite instability", "MSI", "MSI-H",
        "PD-L1 expression", "tumor infiltrating lymphocytes", "TILs",
        "CD8+ T cells", "cytotoxic T cells",
        "T cell exhaustion", "exhausted T cells",
        "T cell dysfunction", "dysfunctional T cells",
        "immune signature", "immunophenotype",
        "immune score", "immunoscore"
    ]
}


In [ ]:
# len(all_icb_keywords)


In [ ]:
all_icb_keywords = ["OS","SD","CR","MSI", "MSI-H", "TMB","B7-H3","IDO1","KIR","avelumab", "Bavencio", "cemiplimab", "Libtayo"]
for category in icb_keywords.values():
    all_icb_keywords.extend(category)
    

In [ ]:
l1_del = []
filtered_keywords = [keyword for keyword in all_icb_keywords if keyword not in l1_del ]


In [ ]:
# gse_gsm_treatment_info.columns


In [ ]:

scRNA_GSEID_df = pd.read_csv('/fse/home/wangyongyan/Project/parser_code/result/2026_01_ICB_wyt/GSE_output.csv',sep=',')
scRNA_GSEID = sorted(scRNA_GSEID_df['GSE'].to_list())


In [ ]:
gse_gsm_treatment_info = pd.read_csv('/fse/home/wangyongyan/Project/parser_code/result/2026_01_ICB_wyt/gse_gsm_meta_rectify.csv',sep=',')
gse_gsm_treatment_info


In [ ]:
gse_gsm_treatment_info = gse_gsm_treatment_info[gse_gsm_treatment_info['gse'].isin(scRNA_GSEID)]
# gse_gsm_treatment_info


In [ ]:
gse_gsm_treatment_info

In [ ]:
import re
import pandas as pd

# 创建正则表达式，不区分大小写
pattern = '|'.join([re.escape(keyword) for keyword in filtered_keywords])
regex_pattern = re.compile(pattern, re.IGNORECASE)
# 定义一个函数检查每行是否包含任何关键词
def contains_keyword(row):
    # 将行中所有值转换为字符串并拼接
    row_text = ' '.join(str(val) for val in row if pd.notna(val))
    return bool(regex_pattern.search(row_text))

# 应用函数到每一行
mask = gse_gsm_treatment_info.apply(contains_keyword, axis=1)
icb_rows = gse_gsm_treatment_info[mask]
print(f"找到 {len(icb_rows)} 行包含ICB关键词的数据")
print(icb_rows.head())


In [ ]:
# icb_rows


In [ ]:
def search_in_specific_columns(df, keywords, search_columns=None):
    if search_columns is None:
        # 默认搜索所有列
        search_columns = df.columns.tolist()
    
    # 创建掩码
    mask = pd.Series(False, index=df.index)
    for col in search_columns:
        if col in df.columns:
            col_mask = df[col].astype(str).str.contains(
                '|'.join(keywords),
                case=False,
                na=False,
                regex=True
            )
            mask = mask | col_mask
    
    result = df[mask].copy()
    # 添加匹配信息
    result['search_columns'] = ', '.join(search_columns)
    return result

# 指定在哪些列中搜索（根据你的列名，这些列可能包含治疗信息）
treatment_columns = ['treat_type', 'treat_final']

icb_rows_specific = search_in_specific_columns(
    gse_gsm_treatment_info, 
   filtered_keywords, 
    treatment_columns
)

print(f"在指定列中找到 {len(icb_rows_specific)} 行包含ICB关键词的数据")


In [ ]:
# len(icb_rows_specific['treat_final'].unique())


In [ ]:
len(icb_rows_specific['gse'].unique())


In [ ]:
# icb_rows_specific[icb_rows_specific['treat_final'].str.contains("")]['treat_final'].unique()



In [ ]:
icb_rows_specific.to_csv('external_inputs/scRNA_ICB_metatable.csv',sep=',',index=False)

In [ ]:
############################ 在这里试下 Squidiff 这个工具
adata = sc.read_h5ad('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_BatT_12608.h5ad')


In [ ]:
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


In [ ]:
import matplotlib.colors as mcolors
hex_colors  = ['#d60909', # EC1
               '#e63d09', # EC2
               '#e66c09', # EC3
               '#e68e09']

In [ ]:
rgb_colors = [mcolors.to_rgb(color) for color in hex_colors]

# Create a custom colormap from these colors
cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", rgb_colors)


fig, axs = plt.subplots(1,1,figsize=(5,4),dpi=800)
for (j,i) in enumerate(np.unique(adata.obs['Celltype_general'])):
    print(i)
    g=axs.scatter(adata.obsm['X_umap'][adata.obs['Celltype_general']==i, 0], 
                  adata.obsm['X_umap'][adata.obs['Celltype_general']==i, 1], 
                  color=hex_colors[j], 
                  alpha=0.9,s=0.6, 
                 )
plt.legend(np.unique(adata.obs['Celltype_general']))
plt.axis('off')


In [ ]:
# sorted(dir())
adata


In [ ]:
fig, axs = plt.subplots(1,1,figsize=(4,3),dpi=800)
g = axs.scatter(adata.obsm['X_umap'][:, 0], 
            adata.obsm['X_umap'][:, 1], 
            c=adata[:,'COL10A1'].to_df().values, 
            alpha=0.9,s=0.5, 
            cmap = 'Blues',
            vmin=0,
            vmax=6
         )
plt.colorbar(g)
#plt.legend(np.unique(subsamp_adata.obs['day']))
plt.axis('off')


In [ ]:
sc.tl.paga(adata, groups="Celltype_general")
